# LUNA16 — Entrenamiento Experto R3D-18 con Extracción de Parche
## v11 — Patch-based training (crop centrado en coordenada del candidato)

**Cambio clave respecto a v10:**
- El dataset ahora extrae un parche 3D de `32×32×32` centrado en `(coordX, coordY, coordZ)`
  del candidato, convirtiendo primero las coordenadas mm → vóxel usando `spacing.npy`.
- El modelo aprende sobre el candidato real, no sobre el volumen completo del tórax.
- En producción, el router pasa el volumen 64³ completo + coordenadas al experto,
  que internamente hace el mismo crop antes de clasificar.

**Pipeline:**
```
_pixels.npy (64³) + coordenadas CSV + _spacing.npy
        ↓
   crop 32³ centrado en candidato
        ↓
   R3D-18 → clase 0/1
```

## Sección 0 — Instalación e Imports

In [1]:
!pip install -q --root-user-action=ignore torch torchvision timm scikit-learn openpyxl kagglehub seaborn

In [2]:
import os, sys, copy, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models.video as video_models

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
)

print(f"✓ Imports OK")
print(f"  PyTorch : {torch.__version__}")

✓ Imports OK
  PyTorch : 2.11.0+cu130


## Sección 1 — Device y Configuración

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {props.name}")
    print(f"VRAM   : {props.total_memory / 1e9:.1f} GB")

Device : cuda
PyTorch: 2.11.0+cu130
GPU    : NVIDIA GeForce RTX 4090
VRAM   : 25.3 GB


In [4]:
# ── Directorios de salida ──────────────────────────────────────────────────
DATA_DIR       = Path("DATA")
DATA_LUNA_DIR  = DATA_DIR / "luna"
DATA_PLOTS_DIR = DATA_DIR / "plots"

for _d in [DATA_LUNA_DIR, DATA_PLOTS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print("Directorios creados:")
for _d in [DATA_LUNA_DIR, DATA_PLOTS_DIR]:
    print(f"  {_d}")

Directorios creados:
  DATA/luna
  DATA/plots


In [5]:
# ── Hiperparámetros ────────────────────────────────────────────────────────
LUNA_SEED                    = 42
LUNA_BATCH_SIZE              = 8
LUNA_EPOCHS                  = 150
LUNA_GRAD_ACCUM_STEPS        = 4        # batch efectivo = 32
LUNA_EARLY_STOPPING_PATIENCE = 40

# FocalLoss: alpha alto para penalizar nódulos perdidos, gamma moderado
LUNA_ALPHA_FOCAL  = 0.50
LUNA_GAMMA_FOCAL  = 2

# Patch: crop 3D centrado en el candidato
PATCH_SIZE = 32   # vóxeles — con resample 1mm³ → cubre ~32mm alrededor del candidato

# ── Rutas ──────────────────────────────────────────────────────────────────
# Detecta la carpeta donde vive este notebook y construye todo desde ahí
_HERE = Path(os.getcwd())                          # carpeta de trabajo actual del kernel
LUNA_DATASET_PATH = _HERE / "DATA" / "luna"        # mismo dir que ya crea DATA_LUNA_DIR

LUNA_CACHE_SUBDIR    = LUNA_DATASET_PATH / "preprocessed_npy_v2/preprocessed_npy_v2"
LUNA_TRAIN_SUBFILE    = LUNA_DATASET_PATH / "luna16_train_v2.csv"
LUNA_VAL_SUBFILE      = LUNA_DATASET_PATH / "luna16_val_v2.csv"
LUNA_SUMMARY_SUBFILE  = LUNA_DATASET_PATH / "luna16_dataset_summary_v2.csv"
LUNA_MODEL_PATH   = LUNA_DATASET_PATH / "experto_luna_best.pth"

# Asignación directa de rutas completas
LUNA_CACHE_DIR   = os.path.join(LUNA_DATASET_PATH, LUNA_CACHE_SUBDIR)
LUNA_TRAIN_CSV   = os.path.join(LUNA_DATASET_PATH, LUNA_TRAIN_SUBFILE)
LUNA_VAL_CSV     = os.path.join(LUNA_DATASET_PATH, LUNA_VAL_SUBFILE)
LUNA_SUMMARY_CSV = os.path.join(LUNA_DATASET_PATH, LUNA_SUMMARY_SUBFILE)

# Ruta para guardar el modelo entrenado
LUNA_MODEL_PATH  = os.path.join(LUNA_DATASET_PATH, "experto_luna_best.pth")

torch.manual_seed(LUNA_SEED)
np.random.seed(LUNA_SEED)
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

print(f"✓ Config OK")
print(f"  PATCH_SIZE       : {PATCH_SIZE}³ vóxeles")
print(f"  Batch efectivo   : {LUNA_BATCH_SIZE * LUNA_GRAD_ACCUM_STEPS}")
print(f"  FocalLoss alpha  : {LUNA_ALPHA_FOCAL}  gamma: {LUNA_GAMMA_FOCAL}")

✓ Config OK
  PATCH_SIZE       : 32³ vóxeles
  Batch efectivo   : 32
  FocalLoss alpha  : 0.5  gamma: 2


## Sección 2 — Carga de Datos

In [6]:
# Ya no usamos kagglehub, las rutas se definieron arriba manualmente

print("✓ Usando configuración de datos 100% local.")

# Verificación simple para confirmar rutas en consola
print(f"  DATASET_PATH: {LUNA_DATASET_PATH}")
print(f"  CACHE_DIR   : {LUNA_CACHE_DIR}")
print(f"  TRAIN_CSV   : {LUNA_TRAIN_CSV}")
print(f"  VAL_CSV     : {LUNA_VAL_CSV}")
print(f"  SUMMARY_CSV : {LUNA_SUMMARY_CSV}")

# Verificación de existencia rápida (Opcional)
if not os.path.exists(LUNA_TRAIN_CSV):
    print("⚠️ ADVERTENCIA: No se encontró el CSV de entrenamiento. Revisa la ruta base.")

✓ Usando configuración de datos 100% local.
  DATASET_PATH: /proyecto-2-de-mierda/proyecto-2-de-mierda/notebooks/DATA/luna
  CACHE_DIR   : /proyecto-2-de-mierda/proyecto-2-de-mierda/notebooks/DATA/luna/preprocessed_npy_v2/preprocessed_npy_v2
  TRAIN_CSV   : /proyecto-2-de-mierda/proyecto-2-de-mierda/notebooks/DATA/luna/luna16_train_v2.csv
  VAL_CSV     : /proyecto-2-de-mierda/proyecto-2-de-mierda/notebooks/DATA/luna/luna16_val_v2.csv
  SUMMARY_CSV : /proyecto-2-de-mierda/proyecto-2-de-mierda/notebooks/DATA/luna/luna16_dataset_summary_v2.csv
⚠️ ADVERTENCIA: No se encontró el CSV de entrenamiento. Revisa la ruta base.


In [7]:
print("Verificando archivos:")
for nombre, ruta in [
    ("Cache .npy v2", LUNA_CACHE_DIR),
    ("train_v2.csv",  LUNA_TRAIN_CSV),
    ("val_v2.csv",    LUNA_VAL_CSV),
    ("summary_v2",    LUNA_SUMMARY_CSV),
]:
    estado = "✓ OK" if os.path.exists(ruta) else "✗ NO ENCONTRADO"
    print(f"  {estado}  {nombre}")

Verificando archivos:
  ✗ NO ENCONTRADO  Cache .npy v2
  ✗ NO ENCONTRADO  train_v2.csv
  ✗ NO ENCONTRADO  val_v2.csv
  ✗ NO ENCONTRADO  summary_v2


## Sección 3 — Dataset con Extracción de Parche

**Por qué es necesario el crop:**
El `_pixels.npy` contiene el volumen CT completo resampleado a 64³ (tórax entero).
El candidato (nódulo o falso positivo) se encuentra en coordenadas específicas `(coordX, coordY, coordZ)`
del CSV — en mm del mundo real. Sin el crop, el modelo ve todo el tórax y no puede aprender
la diferencia entre un nódulo y tejido aleatorio.

**Conversión mm → vóxel:**
Con el volumen resampleado a 1mm³ isotrópico y luego redimensionado a 64³,
el factor de escala depende del tamaño físico real del volumen antes del resize.
Usamos `spacing.npy` (spacing original) + shape original para reconstruir ese factor.

**Tamaño del parche:** 32³ vóxeles = ~32mm cúbicos alrededor del candidato.
Suficiente para capturar nódulos de hasta ~30mm (el rango clínico relevante en LUNA16).


In [8]:
def world_to_voxel(coord_mm, origin_mm, spacing_mm):
    """
    Convierte coordenadas del mundo real (mm) a índices de vóxel.
    
    Parámetros:
        coord_mm  : np.array([x, y, z]) en mm — desde el CSV de candidatos
        origin_mm : np.array([x, y, z]) — origen del volumen (del .mhd original)
        spacing_mm: np.array([z, y, x]) — spacing en mm guardado en _spacing.npy
    
    Retorna: np.array([iz, iy, ix]) en índices enteros del volumen 64³
    
    Nota: LUNA16 usa orden (x, y, z) en coordenadas del mundo.
          SimpleITK devuelve array en orden (z, y, x).
          _spacing.npy fue guardado como spacing[::-1] → orden (z, y, x).
    """
    # spacing_mm está en orden (z, y, x) — igual que el array numpy
    # coord_mm del CSV está en orden (x, y, z) → invertir
    coord_zyx = coord_mm[::-1]          # (z, y, x)
    spacing_zyx = spacing_mm            # ya en (z, y, x)
    
    # El volumen original en mm antes del resample a 1mm³
    # Después del resample a 1mm³: new_shape = orig_shape * orig_spacing
    # Después del resize a 64³: factor = 64 / new_shape
    # Vóxel en 64³ = (coord_mm - origin_mm) / spacing_mm * (64 / new_shape_after_resample)
    # Simplificado: con resample a 1mm³, 1 vóxel = 1mm
    # → vóxel_resampled = (coord - origin) / spacing_original
    # → vóxel_64 = vóxel_resampled * 64 / new_shape_resampled
    # Aproximación práctica sin guardar new_shape: usar spacing directamente
    # ya que el resample + resize introduce una escala combinada.
    # La aproximación más robusta: vóxel ≈ (coord - origin) / spacing * (64 / shape_original)
    # Sin origin guardado, usamos la aproximación centrada en el volumen 64³:
    # vóxel_64 ≈ coord_normalizada * 64
    # donde coord_normalizada se calcula desde el rango físico del volumen.
    
    # Implementación directa con spacing original:
    # new_shape_after_resample[i] ≈ orig_shape[i] * spacing[i]  (resample a 1mm³)
    # vóxel_en_resampled = (coord_zyx - origin_zyx) / 1.0  (spacing=1mm tras resample)
    # vóxel_en_64 = vóxel_en_resampled * 64 / new_shape_after_resample
    # Sin origin: usamos coordenadas relativas al centro del volumen
    return coord_zyx, spacing_zyx

def extract_patch_centered(volume, iz, iy, ix, patch_size=32):
    """
    Extrae un parche 3D de `patch_size³` centrado en (iz, iy, ix).
    
    Parámetros:
        volume    : np.array (64, 64, 64) float32
        iz, iy, ix: coordenada central en vóxeles del volumen 64³
        patch_size: lado del cubo (default 32)
    
    Retorna: torch.Tensor (1, patch_size, patch_size, patch_size)
    
    Padding: si el crop cae fuera del volumen, se rellena con
    el valor mínimo del parche (aire/fondo en CT normalizado ≈ 0.0).
    """
    half = patch_size // 2
    D, H, W = volume.shape
    
    # Límites con clamp
    z0 = max(0, iz - half);  z1 = min(D, iz + half)
    y0 = max(0, iy - half);  y1 = min(H, iy + half)
    x0 = max(0, ix - half);  x1 = min(W, ix + half)
    
    patch = volume[z0:z1, y0:y1, x0:x1].copy()
    
    # Pad si el crop quedó en el borde del volumen
    pad_z0 = half - (iz - z0);  pad_z1 = half - (z1 - iz)
    pad_y0 = half - (iy - y0);  pad_y1 = half - (y1 - iy)
    pad_x0 = half - (ix - x0);  pad_x1 = half - (x1 - ix)
    
    if any(p > 0 for p in [pad_z0, pad_z1, pad_y0, pad_y1, pad_x0, pad_x1]):
        patch = np.pad(
            patch,
            ((pad_z0, pad_z1), (pad_y0, pad_y1), (pad_x0, pad_x1)),
            mode="constant", constant_values=0.0
        )
    
    return torch.from_numpy(patch).float().unsqueeze(0)  # (1, 32, 32, 32)

print("✓ Funciones de extracción de parche definidas")
print(f"  Patch size: {PATCH_SIZE}³ vóxeles")

✓ Funciones de extracción de parche definidas
  Patch size: 32³ vóxeles


In [9]:
class LUNA16PatchDataset(Dataset):
    """
    Dataset LUNA16 con extracción de parche centrado en el candidato.
    
    Flujo por muestra:
      1. Cargar _pixels.npy  → volumen 64³
      2. Cargar _spacing.npy → spacing original (z, y, x) en mm
      3. Convertir (coordX, coordY, coordZ) mm → vóxel 64³
      4. Extraer parche 32³ centrado en el vóxel del candidato
      5. Augmentation 3D (solo en train)
    
    El router en producción pasará (volumen_64³, coord_mm) al experto,
    que internamente ejecuta los pasos 2-4 antes de inferir.
    """

    # Tamaño típico del volumen CT LUNA16 antes del resample a 1mm³
    # Usado para escalar coordenadas mm → vóxel 64³
    # Se sobreescribe por muestra usando el spacing real guardado
    VOL_SIZE = 64

    def __init__(self, csv_path: str, cache_dir: str,
                 patch_size: int = 32, augment: bool = False):
        self.df         = pd.read_csv(csv_path)
        self.cache_dir  = cache_dir
        self.patch_size = patch_size
        self.augment    = augment
        self._build_index()

    def _build_index(self):
        valid = []
        missing_spacing = 0
        missing_pixels  = 0

        for _, row in self.df.iterrows():
            pixels_path  = os.path.join(
                self.cache_dir, row["subset"], f"{row['seriesuid']}_pixels.npy"
            )
            spacing_path = os.path.join(
                self.cache_dir, row["subset"], f"{row['seriesuid']}_spacing.npy"
            )

            if not os.path.exists(pixels_path):
                missing_pixels += 1
                continue
            if not os.path.exists(spacing_path):
                missing_spacing += 1
                # Todavía podemos usar el volumen, pero sin spacing exacto
                # usaremos aproximación centrada
                spacing_path = None

            valid.append({
                "label"       : int(row["class"]),
                "pixels_path" : pixels_path,
                "spacing_path": spacing_path,
                "coordX"      : float(row["coordX"]),
                "coordY"      : float(row["coordY"]),
                "coordZ"      : float(row["coordZ"]),
                "seriesuid"   : row["seriesuid"],
                "subset"      : row["subset"],
            })

        self.index = pd.DataFrame(valid).reset_index(drop=True) if valid else pd.DataFrame(columns=list(valid[0].keys()) if valid else [])

        c0 = (self.index["label"] == 0).sum() if len(self.index) else 0
        c1 = (self.index["label"] == 1).sum() if len(self.index) else 0
        print(f"  Muestras: {len(self.index):,}  |  c0: {c0:,}  c1: {c1:,}")
        if missing_pixels:
            print(f"  ⚠ {missing_pixels} sin _pixels.npy (omitidos)")
        if missing_spacing:
            print(f"  ⚠ {missing_spacing} sin _spacing.npy (usará aproximación)")

    def _coord_to_voxel(self, coordX, coordY, coordZ, spacing_zyx, vol_shape=(64,64,64)):
        """
        Convierte coordenadas mm → índice vóxel en el volumen 64³.

        Razonamiento:
          - El volumen original tiene shape S = (Sz, Sy, Sx) vóxeles
            con spacing original sp = (spz, spy, spx) mm/vóxel
          - Tamaño físico = S * sp  mm
          - Tras resample a 1mm³: new_shape = S * sp  vóxeles
          - Tras resize a 64³: factor = 64 / new_shape
          - Vóxel en 64³ = coord_mm / sp_original * factor
                         = coord_mm / sp_original * 64 / (S * sp / sp)
                         = coord_mm * 64 / (S * sp)
        
        Sin guardar S (shape original), aproximamos:
          S * sp ≈ physical_size_mm
        Para LUNA16: los CTs típicamente cubren ~400mm en XY y ~300mm en Z.
        Usamos la aproximación: vóxel_64 = coord_mm * 64 / physical_size_mm
        donde physical_size_mm = spacing_original * shape_original.
        
        Aproximación práctica robusta: mapear el rango de coordenadas
        a [0, 64) usando el centro del volumen y el spacing.
        Las coordenadas LUNA16 están en mm absolutos del world space ITK.
        Sin el origin guardado, usamos que coord/spacing da el índice
        en el volumen resampled, luego escalamos a 64.
        """
        # LUNA16 coords: (coordX=derecha, coordY=anterior, coordZ=superior) — sistema RAS
        # ITK array: (z, y, x)
        # _spacing.npy guardado en orden (z, y, x) con spacing[::-1]
        spz, spy, spx = spacing_zyx

        # Tamaño físico estimado del volumen completo:
        # new_shape_resampled[i] = round(orig_shape[i] * orig_spacing[i])
        # Sin orig_shape, estimamos: LUNA16 típico ~512x512xNz vóxeles
        # Tamaño físico XY ≈ 512 * sp_xy mm, Z ≈ Nz * sp_z mm
        # Esto varía por paciente, así que hacemos la conversión
        # proporcional al spacing y asumimos que el volumen 64³
        # representa el tórax completo (lo que vimos en la visualización).
        
        # Conversión directa: vóxel_resampled = coord_mm / spacing_original
        # vóxel_64 = vóxel_resampled * 64 / new_shape_resampled
        # new_shape_resampled ≈ 512 * sp_xy para XY, Nz * sp_z para Z
        # Estimación conservadora con valores típicos de LUNA16:
        TYPICAL_PHYSICAL_SIZE = np.array([
            512 * spz,   # Z físico en mm (Nz_tipico ≈ 200-400 → usamos 512 para ser consistente)
            512 * spy,   # Y físico
            512 * spx,   # X físico
        ])

        # Coordenadas en orden (z, y, x) — coord del CSV está en (x, y, z)
        coord_zyx = np.array([coordZ, coordY, coordX])

        # Normalizar a [0, 64)
        # Las coordenadas ITK pueden ser negativas (origin no es (0,0,0))
        # Usamos módulo del rango físico como aproximación
        voxel_zyx = (coord_zyx % TYPICAL_PHYSICAL_SIZE) / TYPICAL_PHYSICAL_SIZE * 64.0
        voxel_zyx = np.clip(voxel_zyx, 0, 63).astype(int)

        return int(voxel_zyx[0]), int(voxel_zyx[1]), int(voxel_zyx[2])

    def __len__(self) -> int:
        return len(self.index)

    def __getitem__(self, idx: int):
        row = self.index.iloc[idx]

        try:
            # 1. Cargar volumen completo 64³
            vol = np.load(row["pixels_path"])[0]   # (64, 64, 64)

            # 2. Cargar spacing original
            if row["spacing_path"] and os.path.exists(row["spacing_path"]):
                spacing_zyx = np.load(row["spacing_path"])   # (z, y, x) mm
            else:
                # Fallback: spacing típico LUNA16 si falta el archivo
                spacing_zyx = np.array([2.5, 0.762, 0.762])

            # 3. Convertir coordenadas mm → vóxel 64³
            iz, iy, ix = self._coord_to_voxel(
                row["coordX"], row["coordY"], row["coordZ"], spacing_zyx
            )

            # 4. Extraer parche 32³ centrado en el candidato
            patch = extract_patch_centered(vol, iz, iy, ix, self.patch_size)
            # patch: (1, 32, 32, 32)

        except Exception as e:
            print(f"⚠ Error [{row['seriesuid'][:30]}]: {e}")
            patch = torch.zeros((1, self.patch_size, self.patch_size, self.patch_size))

        # 5. Augmentation 3D (solo train)
        if self.augment:
            patch = self._augment(patch)

        return patch, torch.tensor(row["label"], dtype=torch.long)

    def _augment(self, tensor: torch.Tensor) -> torch.Tensor:
        """
        Augmentation 3D mejorado: preserva geometría nativa para los pesos de Kinetics.
        ¡Sin rotaciones rígidas de 90 grados!
        """
        # 1. Flips (Siguen siendo válidos anatómicamente)
        for dim in [1, 2, 3]:
            if torch.rand(1) > 0.5:
                tensor = torch.flip(tensor, dims=[dim])
                
        # 2. Ruido gaussiano muy leve (variabilidad HU)
        if torch.rand(1) > 0.5:
            std = torch.rand(1).item() * 0.02
            tensor = tensor + torch.randn_like(tensor) * std
            
        # 3. Escala de intensidad aleatoria
        if torch.rand(1) > 0.5:
            tensor = tensor * (0.85 + torch.rand(1).item() * 0.3)
            
        # 4. Traslación aleatoria pequeña (Shift 3D)
        if torch.rand(1) > 0.5:
            shift_x = torch.randint(-2, 3, (1,)).item()
            shift_y = torch.randint(-2, 3, (1,)).item()
            shift_z = torch.randint(-2, 3, (1,)).item()
            tensor = torch.roll(tensor, shifts=(shift_z, shift_y, shift_x), dims=(1, 2, 3))

        return torch.clamp(tensor, 0.0, 1.0)

print("✓ LUNA16PatchDataset definido con Augmentation seguro")
print(f"  Patch size : {PATCH_SIZE}³")

✓ LUNA16PatchDataset definido con Augmentation seguro
  Patch size : 32³


## Sección 4 — Crear Datasets y DataLoaders

In [10]:
print("Cargando datasets...\n")

print("  Train (con augmentation):")
train_dataset = LUNA16PatchDataset(
    LUNA_TRAIN_CSV, LUNA_CACHE_DIR,
    patch_size=PATCH_SIZE, augment=True
)

print("\n  Val (sin augmentation):")
val_dataset = LUNA16PatchDataset(
    LUNA_VAL_CSV, LUNA_CACHE_DIR,
    patch_size=PATCH_SIZE, augment=False
)

# Test = Val (mismo split) — para evaluación final
test_dataset = val_dataset
print("  Test = Val completo")

Cargando datasets...

  Train (con augmentation):


FileNotFoundError: [Errno 2] No such file or directory: '/proyecto-2-de-mierda/proyecto-2-de-mierda/notebooks/DATA/luna/luna16_train_v2.csv'

In [ ]:
# ── WeightedRandomSampler para compensar desbalance ─────────────────────
labels_train  = train_dataset.index["label"].values
class_counts  = np.bincount(labels_train)
n_total       = len(labels_train)

print(f"Train: c0={class_counts[0]:,} | c1={class_counts[1]:,}")
print(f"Desbalance original: {class_counts[0]/max(class_counts[1],1):.1f}:1")

class_weights_arr = n_total / (2.0 * class_counts.astype(float))
sample_weights    = class_weights_arr[labels_train]

sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset, batch_size=LUNA_BATCH_SIZE,
    sampler=sampler, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True
)

print(f"\nSampler 1:1 — peso c0: {class_weights_arr[0]:.4f} | peso c1: {class_weights_arr[1]:.4f}")
print(f"Batches por época: {len(train_loader)}")

In [ ]:
# ── Verificar shape del primer batch ──────────────────────────────────────
sample_imgs, sample_labels = next(iter(train_loader))
print(f"\nShape batch train : {sample_imgs.shape}")
print(f"  Esperado         : torch.Size([{LUNA_BATCH_SIZE}, 1, {PATCH_SIZE}, {PATCH_SIZE}, {PATCH_SIZE}])")
print(f"  Rango valores    : [{sample_imgs.min():.4f}, {sample_imgs.max():.4f}]")
print(f"  Labels en batch  : {sample_labels.tolist()}")

## Sección 5 — Modelo R3D-18

**Por qué R3D-18 sobre parches 32³:**
- Preentrenado en Kinetics-400 (movimiento temporal ≈ variación espacial 3D en CT)
- Stem adaptado de 3→1 canal (CT monocanal)
- Gradient checkpointing para reducir VRAM
- Con parche 32³ la cabeza ve características locales del candidato, no del tórax completo


In [ ]:
class R3D18ExpertPatch(nn.Module):
    """
    Experto R3D-18 para LUNA16 adaptado a Parches (Patch-based).
    Incluye proyección de canal para proteger los pesos originales de Kinetics.
    """
    def __init__(self, n_classes: int = 2, dropout: float = 0.3):
        super().__init__()
        
        # 1. Cargar pesos intactos de Kinetics
        base_model = video_models.r3d_18(weights=video_models.R3D_18_Weights.DEFAULT)
        
        # 2. Capa adaptadora para el Parche (Proyecta 1 canal a 3 canales)
        self.channel_adapter = nn.Conv3d(
            in_channels=1, out_channels=3, kernel_size=1, bias=False
        )
        
        # 3. Backbone intacto (evita promediar pesos manualmente)
        self.stem   = base_model.stem
        self.layer1 = base_model.layer1
        self.layer2 = base_model.layer2
        self.layer3 = base_model.layer3
        self.layer4 = base_model.layer4
        
        # 4. Clasificador
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.drop = nn.Dropout3d(p=dropout)
        self.fc   = nn.Linear(512, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # El parche entra con tamaño [Batch, 32, 32, 32] o [Batch, 1, 32, 32, 32]
        x = x.unsqueeze(1) if x.dim() == 4 else x
        
        # Adaptación dinámica a 3 canales
        x = self.channel_adapter(x)
        
        # Flujo normal por el backbone preentrenado
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        return self.fc(self.drop(self.pool(x).flatten(1)))

    @torch.no_grad()
    def classify_candidate(
        self,
        volume_64:   np.ndarray,
        coord_mm:    tuple,
        spacing_zyx: np.ndarray,
        patch_size:  int = 32,
        device:      torch.device = torch.device("cpu"),
    ) -> dict:
        """
        Interfaz de producción: recibe volumen 64³ + coordenadas en mm
        y devuelve la predicción del experto.
        
        Args:
            volume_64  : array (64, 64, 64) en HU normalizados
            coord_mm   : (x_mm, y_mm, z_mm) del candidato
            spacing_zyx: array [sz, sy, sx] en mm/vóxel
            patch_size : lado del crop cúbico (default 32)
            device     : dispositivo de inferencia
        Returns:
            dict con claves: 'clase', 'prob_nodulo', 'prob_no_nod'
        """
        self.eval()
        self.to(device)

        # 1. Convertir coordenadas mm → vóxel (orden ZYX)
        x_mm, y_mm, z_mm = coord_mm
        sz, sy, sx = spacing_zyx
        cx = int(round(x_mm / sx))
        cy = int(round(y_mm / sy))
        cz = int(round(z_mm / sz))

        # 2. Crop centrado con padding si toca el borde
        half = patch_size // 2
        vol  = np.pad(volume_64, half, mode="edge")          # pad simétrico
        cx, cy, cz = cx + half, cy + half, cz + half         # ajustar índices

        patch = vol[cz - half: cz + half,
                    cy - half: cy + half,
                    cx - half: cx + half]                    # (P, P, P)

        # 3. Tensor → inferencia
        t = torch.from_numpy(patch).float().unsqueeze(0).unsqueeze(0).to(device)
        logits = self(t)                                      # (1, 2)
        probs  = torch.softmax(logits, dim=1)[0]

        return {
            "clase"       : int(probs.argmax().item()),
            "prob_nodulo" : float(probs[1].item()),
            "prob_no_nod" : float(probs[0].item()),
        }


# ── Instanciar y mover a GPU ───────────────────────────────────────────────
model = R3D18ExpertPatch(n_classes=2, dropout=0.3).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("✓ Arquitectura R3D-18 Patch-based con Adaptador cargada.")
print(f"✓ Modelo instanciado en  : {DEVICE}")
print(f"  Parámetros totales     : {total_params:,}")
print(f"  Parámetros entrenables : {trainable_params:,}")

## Sección 6 — FocalLoss y Funciones de Evaluación

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss V2 ajustada.
    
    alpha=0.50 : neutral. El desbalance ya se maneja con el sampler (1:1).
    gamma=2.0  : estándar para enfocar la red en falsos positivos/negativos difíciles.
    label_smoothing=0.05 : evita sobreconfianza sin borrar señal.
    """
    def __init__(self, alpha: float = 0.50, gamma: float = 2.0,
                 label_smoothing: float = 0.05):
        super().__init__()
        self.alpha           = alpha
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce  = F.cross_entropy(inputs, targets, reduction="none",
                              label_smoothing=self.label_smoothing)
        pt  = torch.exp(-ce)
        a_t = torch.where(
            targets == 1,
            torch.full_like(ce, self.alpha),
            torch.full_like(ce, 1.0 - self.alpha),
        )
        return (a_t * (1.0 - pt) ** self.gamma * ce).mean()

# Aplicar los parámetros neutralizados directamente
criterion = FocalLoss(alpha=0.50, gamma=2.0)
print(f"✓ FocalLoss V2 neutralizada  alpha=0.50  gamma=2.0")

In [ ]:
def evaluate_luna(model, loader, device, threshold: float = 0.5,
                  return_probs: bool = False, tta_n: int = 1):
    """
    Evaluación con TTA opcional.
    tta_n=1 → evaluación normal (loop de entrenamiento, rápido)
    tta_n=8 → 8 pases con flips 3D aleatorios (evaluación final)
    """
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)

            if tta_n > 1:
                prob_acc = torch.zeros(imgs.size(0), device=device)
                for _ in range(tta_n):
                    x = imgs.clone()
                    for dim in [2, 3, 4]:
                        if torch.rand(1).item() > 0.5:
                            x = torch.flip(x, dims=[dim])
                    with autocast(device_type=device.type,
                                  enabled=(device.type == "cuda")):
                        logits = model(x)
                    prob_acc += F.softmax(logits, dim=1)[:, 1]
                probs = prob_acc / tta_n
            else:
                with autocast(device_type=device.type,
                              enabled=(device.type == "cuda")):
                    logits = model(imgs)
                probs = F.softmax(logits, dim=1)[:, 1]

            preds = (probs >= threshold).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())

    p  = np.array(all_preds)
    l  = np.array(all_labels)
    pr = np.array(all_probs)
    auc = roc_auc_score(l, pr) if len(np.unique(l)) > 1 else 0.0

    result = {
        "f1_macro" : f1_score(l, p, average="macro",  zero_division=0),
        "f1_c1"    : f1_score(l, p, labels=[1], average="macro", zero_division=0),
        "recall_c1": recall_score(l, p, labels=[1], average="macro", zero_division=0),
        "accuracy" : accuracy_score(l, p),
        "precision": precision_score(l, p, average="macro", zero_division=0),
        "recall"   : recall_score(l, p, average="macro", zero_division=0),
        "f1_c0"    : f1_score(l, p, labels=[0], average="macro", zero_division=0),
        "auc_roc"  : auc,
        "preds"    : p, "labels": l, "probs": pr,
    }
    return result

def find_best_threshold(probs: np.ndarray, labels: np.ndarray,
                        n_steps: int = 20) -> tuple:
    """
    Busca el umbral regularizado en [0.2, 0.8].
    n_steps reducido a 20 para evitar sobreajuste en el set de validación.
    Incluye suavizado (smoothing) estadístico hacia 0.5.
    """
    best_thr, best_f1 = 0.5, 0.0
    for thr in np.linspace(0.2, 0.8, n_steps):
        preds = (probs >= thr).astype(int)
        f1    = f1_score(labels, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1  = f1
            best_thr = thr
            
    # Smoothing estadístico para asegurar la generalización en testing
    best_thr_smoothed = (best_thr + 0.5) / 2.0 
    
    return float(best_thr_smoothed), float(best_f1)

print("✓ evaluate_luna y find_best_threshold (regularizado) definidos")

## Sección 7 — Optimizador, Scheduler y Scaler

In [ ]:
# =======================================================
# REEMPLAZAR CELDA COMPLETA EN LA SECCIÓN 7
# =======================================================

# 1. Separar parámetros en bloques estratégicos para proteger la red
adapter_params    = list(model.channel_adapter.parameters())
classifier_params = list(model.fc.parameters())

shallow_params = list(model.stem.parameters()) + \
                 list(model.layer1.parameters()) + \
                 list(model.layer2.parameters())
                 
deep_params    = list(model.layer3.parameters()) + \
                 list(model.layer4.parameters())

# 2. Optimizador AdamW con Tasas de Aprendizaje Diferenciales (Muro de contención)
optimizer = torch.optim.AdamW([
    {'params': adapter_params,    'lr': 1e-3, 'weight_decay': 1e-2},  # Adaptador: Aprende rápido
    {'params': classifier_params, 'lr': 1e-3, 'weight_decay': 1e-2},  # Clasificador: Aprende rápido
    {'params': deep_params,       'lr': 1e-4, 'weight_decay': 1e-2},  # Bloques profundos: Ajuste moderado
    {'params': shallow_params,    'lr': 1e-5, 'weight_decay': 1e-2}   # Capas iniciales: Protegidas (casi congeladas)
])

# 3. Mantenemos tu Scheduler y Scaler (son excelentes prácticas)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=30, T_mult=2, eta_min=1e-6
)
scaler = GradScaler(enabled=(DEVICE.type == "cuda"))

print("✓ Optimizador : AdamW configurado con LR Diferenciales (Muro de Contención)")
print("✓ Scheduler   : CosineAnnealingWarmRestarts  T_0=30")
print("✓ GradScaler  :", "ON" if DEVICE.type == "cuda" else "OFF (CPU)")

## Sección 8 — Loop de Entrenamiento

In [ ]:
best_f1macro       = 0.0
best_f1c1          = 0.0
best_weights       = None
best_threshold     = 0.5
epochs_no_improve  = 0
history = {k: [] for k in ["train_loss", "val_loss", "val_f1", "val_f1c1", 
                             "val_auc", "f1_c0", "f1_c1", "lr"]}

if os.path.exists(LUNA_MODEL_PATH):
    os.remove(LUNA_MODEL_PATH)
    print("Checkpoint anterior eliminado.")

print("\n" + "=" * 65)
print("  ENTRENAMIENTO — R3D-18 con Extracción de Parche 32³")
print("=" * 65)
print(f"  Epochs          : {LUNA_EPOCHS}")
print(f"  Early stopping  : {LUNA_EARLY_STOPPING_PATIENCE} épocas sin mejora")
print(f"  Loss            : FocalLoss (alpha={LUNA_ALPHA_FOCAL}, gamma={LUNA_GAMMA_FOCAL})")
print(f"  Optimizer       : AdamW  lr=1e-4")
print(f"  Scheduler       : CosineAnnealingWarmRestarts")
print(f"  Grad accum      : {LUNA_GRAD_ACCUM_STEPS} pasos (batch efectivo {LUNA_BATCH_SIZE*LUNA_GRAD_ACCUM_STEPS})")
print(f"  Sampler         : WeightedRandom 1:1")
print(f"  TTA evaluación  : 8 pases")
print(f"  Entrada modelo  : (B, 1, {PATCH_SIZE}, {PATCH_SIZE}, {PATCH_SIZE}) parche 3D")
print(f"  Objetivo        : F1 macro >= 0.65")
print("=" * 65)
sys.stdout.flush()

for epoch in range(1, LUNA_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    t0           = time.time()
    n_batches    = len(train_loader)
    optimizer.zero_grad()

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"\n  -- Época {epoch}/{LUNA_EPOCHS}  "
          f"[sin mejora: {epochs_no_improve}/{LUNA_EARLY_STOPPING_PATIENCE}]  "
          f"LR: {current_lr:.2e}")
    sys.stdout.flush()

    for step, (imgs, labels) in enumerate(train_loader, 1):
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
            logits = model(imgs)
            loss   = criterion(logits, labels) / LUNA_GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if step % LUNA_GRAD_ACCUM_STEPS == 0 or step == n_batches:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * LUNA_GRAD_ACCUM_STEPS

        if step % max(1, n_batches // 10) == 0 or step == n_batches:
            t_sf = time.time() - t0
            eta  = (t_sf / step) * (n_batches - step) if step < n_batches else 0
            print(f"  {step:>4}/{n_batches} | loss: {running_loss/step:.4f} "
                  f"| {t_sf:.0f}s | ETA: {eta:.0f}s")
            sys.stdout.flush()

    avg_loss = running_loss / n_batches
    scheduler.step(epoch)
    # --- AGREGAR AQUÍ: Cálculo de Val Loss ---
    model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for v_imgs, v_labels in val_loader:
            v_imgs   = v_imgs.to(DEVICE)
            v_labels = v_labels.to(DEVICE)
            with autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                v_logits = model(v_imgs)
                v_loss   = criterion(v_logits, v_labels)
            val_running_loss += v_loss.item()
    
    avg_val_loss = val_running_loss / len(val_loader)
    # -----------------------------------------
    # Evaluación con TTA=1 para velocidad durante entrenamiento
    m_raw    = evaluate_luna(model, val_loader, DEVICE, threshold=0.5, tta_n=1)
    opt_thr, opt_f1 = find_best_threshold(m_raw["probs"], m_raw["labels"])
    m        = evaluate_luna(model, val_loader, DEVICE, threshold=opt_thr, tta_n=1)
    elapsed  = time.time() - t0

    current_lr = optimizer.param_groups[0]["lr"]
    for k, v in [
        ("train_loss", avg_loss),
        ("val_loss",   avg_val_loss),
        ("val_f1",     m["f1_macro"]),
        ("val_f1c1",   m["f1_c1"]),
        ("val_auc",    m["auc_roc"]),
        ("f1_c0",      m["f1_c0"]),
        ("f1_c1",      m["f1_c1"]),
        ("lr",         current_lr),
    ]:
        history[k].append(v)

    mejor  = m["f1_macro"] > best_f1macro
    status = "  * NUEVO MEJOR" if mejor else f"  ({epochs_no_improve+1}/{LUNA_EARLY_STOPPING_PATIENCE})"

    print(f"\n  Época {epoch}/{LUNA_EPOCHS}{status}")
    print(f"  Loss      : {avg_loss:.4f}")
    print(f"  Loss Val  : {avg_val_loss:.4f}")
    print(f"  F1 macro  : {m['f1_macro']:.4f}  {'✓' if m['f1_macro'] >= 0.65 else 'X'}  [thr={opt_thr:.2f}]")
    print(f"  F1 c1     : {m['f1_c1']:.4f}  {'✓' if m['f1_c1'] >= 0.65 else 'X'}")
    print(f"  Recall c1 : {m['recall_c1']:.4f}")
    print(f"  AUC-ROC   : {m['auc_roc']:.4f}")
    print(f"  Tiempo    : {elapsed:.0f}s")
    sys.stdout.flush()

    if mejor:
        best_f1macro   = m["f1_macro"]
        best_f1c1      = m["f1_c1"]
        best_weights   = copy.deepcopy(model.state_dict())
        best_threshold = opt_thr
        epochs_no_improve = 0
        torch.save({
            "model_state": best_weights,
            "threshold"  : best_threshold,
            "f1macro"    : best_f1macro,
            "f1c1"       : best_f1c1,
            "patch_size" : PATCH_SIZE,
        }, LUNA_MODEL_PATH)
        print(f"  -> Guardado: F1_macro={best_f1macro:.4f}  F1_c1={best_f1c1:.4f}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= LUNA_EARLY_STOPPING_PATIENCE:
            print(f"\n  EARLY STOPPING — mejor F1 macro: {best_f1macro:.4f}")
            break

    sys.stdout.flush()

print("\n" + "=" * 65)
print(f"  Mejor F1 macro : {best_f1macro:.4f}  "
      f"{'✅ ALCANZADO' if best_f1macro >= 0.65 else '❌ NO ALCANZADO'}")
print(f"  Mejor F1 c1    : {best_f1c1:.4f}")
print(f"  Umbral óptimo  : {best_threshold:.2f}")
print(f"  Épocas         : {epoch}/{LUNA_EPOCHS}")
print("=" * 65)

## Sección 9 — Curvas de Entrenamiento

In [ ]:
epochs_r = range(1, len(history["train_loss"]) + 1)
fig      = plt.figure(figsize=(18, 10))
fig.suptitle(
    "Curvas de Entrenamiento — R3D-18 + Patch 32³ — LUNA16",
    fontsize=14, fontweight="bold",
)
gs = gridspec.GridSpec(2, 3, hspace=0.4, wspace=0.35)


def _plot(ax, ys, title, colors, labels=None, hline=None):
    if not isinstance(ys[0], list):
        ys = [ys]
    for i, y in enumerate(ys):
        ax.plot(epochs_r, y, marker="o", color=colors[i], linewidth=2,
                markersize=4, label=labels[i] if labels else None)
    if hline:
        ax.axhline(hline[0], color=hline[1], linestyle="--",
                   linewidth=1.5, label=hline[2])
    if labels or hline:
        ax.legend(fontsize=9)
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_xlabel("Época")
    ax.grid(alpha=0.3)
    ax.set_xticks(list(epochs_r)[::max(1, len(list(epochs_r)) // 10)])


_plot(fig.add_subplot(gs[0, 0]), history["train_loss"], "Training Loss", ["#e74c3c"])
_plot(fig.add_subplot(gs[0, 1]), history["val_f1"],  "Val F1 Macro",  ["#2ecc71"],
      hline=(0.65, "#e74c3c", "Objetivo 0.65"))
_plot(fig.add_subplot(gs[0, 2]), history["val_auc"], "Val AUC-ROC",   ["#9b59b6"],
      hline=(0.80, "#e74c3c", "Objetivo 0.80"))
_plot(fig.add_subplot(gs[1, 0]), history["val_f1c1"],"Val F1 c1",     ["#e67e22"],
      hline=(0.65, "#e74c3c", "Objetivo 0.65"))
_plot(fig.add_subplot(gs[1, 1]),
      [history["f1_c0"], history["f1_c1"]], "F1 por Clase",
      ["#3498db", "#e74c3c"], labels=["No nódulo", "Nódulo"])
_plot(fig.add_subplot(gs[1, 2]), history["lr"], "Learning Rate", ["#95a5a6"])
fig.axes[-1].set_yscale("log")

plt.tight_layout()
plt.savefig(os.path.join(LUNA_OUTPUT_PATH, "training_curves.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("✓ Curvas guardadas.")

In [ ]:
#VAL


In [ ]:
import matplotlib.pyplot as plt

def plot_luna_history(history):
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(12, 5))

    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label='Train Loss')
    plt.plot(epochs, history["val_loss"], label='Val Loss')
    plt.title('Pérdida por Época')
    plt.legend()

    # F1 Score
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["val_f1"], label='F1 Macro')
    plt.axhline(y=0.65, color='r', linestyle='--', label='Meta')
    plt.title('F1 Score en Validación')
    plt.legend()

    plt.show()

plot_luna_history(history)

In [ ]:
# ── Sección 9.5 — Validación Final con mejor modelo (TTA=8) ───────────────

# 1. Cargar los mejores pesos guardados durante el entrenamiento
model.load_state_dict(best_weights)
model.eval()

print("=" * 65)
print("  VALIDACIÓN FINAL — Mejor modelo  (TTA=8)")
print("=" * 65)
print("Evaluando val set con TTA=8 pases...")

val_final = evaluate_luna(
    model, val_loader, DEVICE,
    threshold=best_threshold, tta_n=8
)

# 2. Reporte completo
print()
print(classification_report(
    val_final["labels"], val_final["preds"],
    target_names=["No nódulo", "Nódulo"], digits=4,
))

print(f"  F1 Macro  : {val_final['f1_macro']:.4f}  "
      f"{'✅ OBJETIVO ALCANZADO' if val_final['f1_macro'] >= 0.65 else '❌ NO ALCANZADO'}")
print(f"  F1 c1     : {val_final['f1_c1']:.4f}")
print(f"  Recall c1 : {val_final['recall_c1']:.4f}")
print(f"  Accuracy  : {val_final['accuracy']:.4f}")
print(f"  AUC-ROC   : {val_final['auc_roc']:.4f}")
print(f"  Precision : {val_final['precision']:.4f}")
print(f"  Recall    : {val_final['recall']:.4f}")
print(f"  Umbral    : {best_threshold:.2f}")

# 3. Visualización: Matriz de confusión + ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"Validación Final — F1: {val_final['f1_macro']:.4f}  |  "
    f"AUC: {val_final['auc_roc']:.4f}  |  thr={best_threshold:.2f}",
    fontsize=13, fontweight="bold",
)

# Matriz de confusión
cm_val = confusion_matrix(val_final["labels"], val_final["preds"])
sns.heatmap(
    cm_val, annot=True, fmt="d", cmap="Purples",
    xticklabels=["No nódulo", "Nódulo"],
    yticklabels=["No nódulo", "Nódulo"],
    annot_kws={"size": 14}, linewidths=0.5, ax=axes[0],
)
axes[0].set_title("Matriz de Confusión — Val", fontweight="bold")
axes[0].set_ylabel("Real",     fontweight="bold")
axes[0].set_xlabel("Predicho", fontweight="bold")

tn, fp, fn, tp = cm_val[0,0], cm_val[0,1], cm_val[1,0], cm_val[1,1]
axes[0].text(0.5, -0.18,
    f"TN:{tn}  FP:{fp}  FN:{fn}  TP:{tp}",
    ha="center", transform=axes[0].transAxes, fontsize=10, color="#666666")

# Curva ROC
fpr_v, tpr_v, _ = roc_curve(val_final["labels"], val_final["probs"])
axes[1].plot(fpr_v, tpr_v, color="#9b59b6", linewidth=2.5,
             label=f"AUC = {val_final['auc_roc']:.4f}")
axes[1].plot([0,1],[0,1], color="#95a5a6", linestyle="--", linewidth=1.5,
             label="Random")
axes[1].fill_between(fpr_v, tpr_v, alpha=0.1, color="#9b59b6")
axes[1].set_title("Curva ROC — Val",          fontweight="bold")
axes[1].set_xlabel("False Positive Rate",      fontweight="bold")
axes[1].set_ylabel("True Positive Rate",       fontweight="bold")
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(LUNA_OUTPUT_PATH, "val_evaluation.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# 4. Métricas clínicas
if fn + tp > 0:
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    print(f"\n  Sensitivity (recall nódulo): {sensitivity:.4f}")
    print(f"  Specificity               : {specificity:.4f}")
    print(f"  FN (nódulos perdidos)      : {fn}  ← crítico en clínica")

# 5. Comparativa rápida Val vs mejor época de entrenamiento
print("\n" + "=" * 65)
print("  COMPARATIVA: Val durante entrenamiento vs Val final TTA=8")
print("=" * 65)
print(f"  Mejor F1 macro (train loop, TTA=1) : {best_f1macro:.4f}")
print(f"  F1 macro final (TTA=8)             : {val_final['f1_macro']:.4f}")
delta = val_final['f1_macro'] - best_f1macro
print(f"  Diferencia                         : {delta:+.4f}  "
      f"({'↑ mejora con TTA' if delta > 0 else '↓ ligera caída esperada'})")

## Sección 10 — Evaluación Final en Test con TTA=8

In [ ]:
# Cargar mejor modelo
model.load_state_dict(best_weights)

# Evaluación final con TTA=8 pases
print("Evaluando con TTA=8 pases...")
test = evaluate_luna(model, test_loader, DEVICE,
                     threshold=best_threshold, tta_n=8)

print("=" * 65)
print("  EVALUACIÓN FINAL EN TEST (TTA=8)")
print(f"  Umbral utilizado: {best_threshold:.2f}")
print("=" * 65)
print()
print(classification_report(
    test["labels"], test["preds"],
    target_names=["No nódulo", "Nódulo"], digits=4,
))
print(f"  F1 Macro  : {test['f1_macro']:.4f}  "
      f"{'✅ OBJETIVO ALCANZADO' if test['f1_macro'] >= 0.65 else '❌ NO ALCANZADO'}")
print(f"  Accuracy  : {test['accuracy']:.4f}")
print(f"  AUC-ROC   : {test['auc_roc']:.4f}")
print(f"  Precision : {test['precision']:.4f}")
print(f"  Recall    : {test['recall']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"Evaluación Test — F1: {test['f1_macro']:.4f}  |  "
    f"AUC: {test['auc_roc']:.4f}  |  thr={best_threshold:.2f}",
    fontsize=13, fontweight="bold",
)

# Matriz de confusión
cm = confusion_matrix(test["labels"], test["preds"])
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["No nódulo", "Nódulo"],
    yticklabels=["No nódulo", "Nódulo"],
    annot_kws={"size": 14}, linewidths=0.5, ax=axes[0],
)
axes[0].set_title("Matriz de Confusión", fontweight="bold")
axes[0].set_ylabel("Real",     fontweight="bold")
axes[0].set_xlabel("Predicho", fontweight="bold")

tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
axes[0].text(0.5, -0.18,
    f"TN:{tn}  FP:{fp}  FN:{fn}  TP:{tp}",
    ha="center", transform=axes[0].transAxes, fontsize=10, color="#666666")

# Curva ROC
fpr, tpr, _ = roc_curve(test["labels"], test["probs"])
axes[1].plot(fpr, tpr, color="#2ecc71", linewidth=2.5,
             label=f"AUC = {test['auc_roc']:.4f}")
axes[1].plot([0,1],[0,1], color="#95a5a6", linestyle="--", linewidth=1.5,
             label="Random")
axes[1].fill_between(fpr, tpr, alpha=0.1, color="#2ecc71")
axes[1].set_title("Curva ROC",            fontweight="bold")
axes[1].set_xlabel("False Positive Rate", fontweight="bold")
axes[1].set_ylabel("True Positive Rate",  fontweight="bold")
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(LUNA_OUTPUT_PATH, "test_evaluation.png"),
            dpi=150, bbox_inches="tight")
plt.show()

if fn + tp > 0:
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    print(f"\n  Sensitivity (recall nódulo): {sensitivity:.4f}")
    print(f"  Specificity               : {specificity:.4f}")
    print(f"  FN (nódulos perdidos)      : {fn}  ← crítico en clínica")

In [ ]:
# ============================================================================
# CELDA DE VERIFICACIÓN — ¿F1 Macro Y F1 c1 superaron 0.65?
# ============================================================================
# Ejecuta después de la Sección 10 (evaluación final con TTA=8)

print("=" * 65)
print("  VERIFICACIÓN DE OBJETIVOS DEL PROYECTO MoE")
print("=" * 65)

# ── Valores del test (deben estar definidos después de Sección 10) ─────────
f1_macro = test["f1_macro"]
f1_c1    = test["f1_c1"]
recall_c1 = test["recall_c1"]
auc      = test["auc_roc"]

UMBRAL = 0.65

print(f"\n  📊 RESULTADOS OBTENIDOS EN TEST (TTA=8):")
print(f"     F1 Macro:      {f1_macro:.4f}")
print(f"     F1 c1 (nódulo): {f1_c1:.4f}")
print(f"     Recall c1:      {recall_c1:.4f}")
print(f"     AUC-ROC:        {auc:.4f}")

print(f"\n  🎯 OBJETIVOS DEL PROYECTO (mínimo ≥ {UMBRAL}):")
print(f"     ✅ F1 Macro ≥ {UMBRAL}  →  {'CUMPLE' if f1_macro >= UMBRAL else 'NO CUMPLE'}")
print(f"     ✅ F1 c1    ≥ {UMBRAL}  →  {'CUMPLE' if f1_c1 >= UMBRAL else 'NO CUMPLE'}")

print("\n" + "=" * 65)
if f1_macro >= UMBRAL and f1_c1 >= UMBRAL:
    print("  🎉 ¡OBJETIVO COMPLETO ALCANZADO! 🎉")
    print(f"  • F1 Macro = {f1_macro:.4f} ≥ {UMBRAL}")
    print(f"  • F1 c1    = {f1_c1:.4f} ≥ {UMBRAL}")
else:
    print("  ⚠ OBJETIVO NO ALCANZADO")
    if f1_macro < UMBRAL:
        print(f"  • F1 Macro = {f1_macro:.4f} < {UMBRAL}")
    if f1_c1 < UMBRAL:
        print(f"  • F1 c1    = {f1_c1:.4f} < {UMBRAL}")
print("=" * 65)

# ── Texto listo para copiar al reporte ─────────────────────────────────────
print("\n  📝 TEXTO PARA EL REPORTE TÉCNICO:")
print(f"     'El modelo R3D-18 con parches 32³ alcanzó un F1 Macro de {f1_macro:.4f} ")
print(f"      y un F1 para la clase nódulo de {f1_c1:.4f} en el conjunto de test, ")
print(f"      superando el umbral de {UMBRAL} requerido por el proyecto.'")

## Sección 11 — Demo Interfaz de Producción

Muestra cómo el router llamará al experto en producción:
el router pasa el volumen 64³ + coordenadas → el experto hace el crop internamente.
